# Root finding

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/root_finding/root_finding.ipynb)

The roots (*aka zeros*) of a function are values of function arguments forwhich the function is zero:

Find $x$ such that:

$$ f(x) = 0 $$

It can become complicated when we consider vector $\mathbf{x}$ and even $\mathbf{f}$, but consider a special case of finding the roots of $\mathbf{f}(\mathbf{x})$ is our familiar linear system, $\mathbf{A} \mathbf{x} -\mathbf{b} = \mathbf{0}$. Solving linear systems is a special case of root finding, which is generalized to nonlinear functions.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import root

try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

## Roots of some nonlinear functions

Let's build some intuition by exploring some type of roots in 1D functions using the *graphical method*: Plot the function and examine where it crosses the x-axis.

NB: Note the structure of the code below - Since we don't know *a priori* where the roots will be, we have to take a series of initial guesses and cross our finger....  and even then we may fail to find them all!

In [ ]:
def plot_and_find_roots(func):
    x = np.linspace(-10, 10, 400)
    y = func(x)

    # Find roots
    x0s = np.arange(-10, 10, 1)
    roots = []
    for x0 in x0s:
        r = root(func, x0=x0)
        if r.success:
            roots.append((r.x[0], r.fun[0]))

    # Remove duplicate roots (within tolerance)
    unique_roots = []
    tol = 1e-6
    for rx, ry in roots:
        if not any(abs(rx - ux) < tol for ux, _ in unique_roots):
            unique_roots.append((rx, ry))

    # Plot with Plotly
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name='f(x)'))
    fig.add_hline(y=0, line_dash='dash', line_color='black', annotation_text='x-axis')
    if unique_roots:
        rx_vals = [rx for rx, _ in unique_roots]
        fig.add_trace(go.Scatter(x=rx_vals, y=[0]*len(unique_roots), mode='markers', marker=dict(color='red', size=10), name='Roots'))
    fig.update_layout(title='Plot of f(x) and its Roots', xaxis_title='x', yaxis_title='f(x)', xaxis_range=[-10, 10], yaxis_range=[-10, 10])
    fig.show()

## Example 1: Real roots - $x^2-4$

In [ ]:
plot_and_find_roots(lambda x: x**2-4)

## Example 2: No roots - $1/x$

(Noting that the vertical line is a plotting artifact).

In [ ]:
plot_and_find_roots(lambda x: 1/x)

## Example 3: Infinite roots $\sin(x^2)$

Only the roots closest to the initial guesses are found!

In [ ]:
plot_and_find_roots(lambda x: np.sin(x**2))

## Complex roots - $x^2+1$

Even the graphical method is not completely reliable due to the existence of *complex roots*.

**But this is wrong!** The quadratic has 2 roots but we need to use a different method.

In [ ]:
def fun(x):
    return x**2 + 1

# Wrong!
plot_and_find_roots(fun)

# Find roots using krylov
root(lambda x: x**2+1, x0 = [1+1j, 1-1j], method = "krylov")

## Visualizing complex roots

In [ ]:
def complex_plot(func):
    real_range = np.linspace(-3, 3, 100)
    imag_range = np.linspace(-3, 3, 100)
    real_part = np.empty((len(real_range), len(imag_range)))

    for i, real in enumerate(real_range):
        for j, imag in enumerate(imag_range):
            z = complex(real, imag)
            result = func(z)
            real_part[i, j] = result.real

    # Find roots
    r = root(lambda x: x**2 + 1, x0=[1 + 1j, 1 - 1j], method="krylov")
    root_x = [root_val.real for root_val in r.x] if r.success else []
    root_y = [root_val.imag for root_val in r.x] if r.success else []

    fig = go.Figure(data=go.Contour(x=real_range, y=imag_range, z=real_part.T, colorscale='Viridis', colorbar=dict(title='Real Part')))
    if root_x:
        fig.add_trace(go.Scatter(x=root_x, y=root_y, mode='markers', marker=dict(color='red', size=10), name='Roots'))
    fig.update_layout(title='Complex Plot of f(z)', xaxis_title='Real', yaxis_title='Imaginary')
    fig.show()

# Use the function to plot x**2 + 1
complex_plot(lambda z: z**2 + 1)

## 3D surface plot of $|f(z)|$

DON'T WORRY - We won't be dealing with complex numbers in general in this course :-)

In [ ]:
def complex_surface_plot(func):
    real_range = np.linspace(-3, 3, 100)
    imag_range = np.linspace(-3, 3, 100)
    abs_val = np.empty((len(real_range), len(imag_range)))

    for i, real in enumerate(real_range):
        for j, imag in enumerate(imag_range):
            z = complex(real, imag)
            result = func(z)
            abs_val[i, j] = abs(result)

    fig = go.Figure(data=[go.Surface(z=abs_val.T, x=real_range, y=imag_range, colorscale='Viridis')])
    fig.update_layout(title='3D Surface Plot of |f(z)|', scene=dict(xaxis_title='Real', yaxis_title='Imaginary', zaxis_title='|f(z)|'))
    fig.show()

# Example: 3D surface plot for x**2 + 1
complex_surface_plot(lambda z: z**2 + 1)

## Pathological / well-behaved functions

Some functions can be very tricky indeed, such as the Weierstrass Function,

$$
\begin{aligned}
\sum_{n=0}^\infty a^n \cos (b^n \pi x) \\
0 < a < 1 \\
a b > 1+\frac{3 \pi}{2}
\end{aligned}
$$

which looks like:

![Weierstrass Function](images/weierstrassfunction.png)

This is a classic example of a continuous but **nowhere differentiable** function! It is a fractal, and yet roots do exist and may be on interest. It is a classic example of a *pathological* function. 

Different applications will have different criteria for *pathological* vs *well behaved* functions, and no general definition exists although there is seldom disagreement among non-mathematicians about which is which. 

Generally, well behaved functions don't have sharp corners, singularities, discontinuities, etc.